# 03 — Gold Layer: Finance

This notebook builds the Finance Gold layer from the cleaned Silver tables.

**Grain**
- `fact_lineitem`: one row per `(order_key, line_number)`.
- `dim_customer`, `dim_supplier`, `dim_part`: one row per business entity.
- `dim_date`: one row per order date.

**Metric definitions used consistently**
- `gross_revenue = extended_price`
- `net_revenue = extended_price * (1 - discount)`
- `net_revenue_with_tax = extended_price * (1 - discount) * (1 + tax)`
- `AOV = SUM(net_revenue) / COUNT(DISTINCT order_key)`

The notebook answers all four Finance questions and leaves reusable Gold views for the demo/presentation.


In [ ]:
# Parameters: keep naming portable across workspaces.
dbutils.widgets.text("catalog", "group1_finance", "Catalog")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")

CATALOG = dbutils.widgets.get("catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

from pyspark.sql import functions as F
from decimal import Decimal

print(f"Source: {SILVER}")
print(f"Target: {GOLD}")


In [ ]:
# Pre-flight checks: Gold depends on the clean Silver tables.
required_silver_tables = ["customer", "supplier", "part", "orders", "lineitem"]

missing = [
    table_name
    for table_name in required_silver_tables
    if not spark.catalog.tableExists(f"{SILVER}.{table_name}")
]

assert not missing, (
    "Missing Silver tables: "
    + ", ".join(missing)
    + ". Run the Silver pipeline before 03_gold."
)

# The Gold schema should have been created by 00_setup.
try:
    spark.sql(f"DESCRIBE SCHEMA {GOLD}")
except Exception as exc:
    raise RuntimeError(
        f"Gold schema {GOLD} does not exist. "
        "Run/coordinate 00_setup before this notebook."
    ) from exc

print("Pre-flight checks passed.")


## 1. Build the Gold star schema

The dimensions keep descriptive attributes out of the fact table.  
`fact_lineitem` joins `silver.lineitem` to `silver.orders` only to attach the customer and order date to each line item.

The fact table deliberately does **not** repeat `o_totalprice` on every line, because summing a repeated order-header total would double-count orders.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_customer
COMMENT 'Finance customer dimension'
AS
SELECT
    c_custkey    AS customer_key,
    c_name       AS customer_name,
    c_mktsegment AS market_segment,
    c_nationkey  AS nation_key,
    c_acctbal    AS account_balance
FROM {SILVER}.customer
""")

print(f"Created {GOLD}.dim_customer")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_supplier
COMMENT 'Finance supplier dimension'
AS
SELECT
    s_suppkey   AS supplier_key,
    s_name      AS supplier_name,
    s_nationkey AS nation_key,
    s_acctbal   AS account_balance
FROM {SILVER}.supplier
""")

print(f"Created {GOLD}.dim_supplier")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_part
COMMENT 'Finance part/product dimension'
AS
SELECT
    p_partkey     AS part_key,
    p_name        AS part_name,
    p_mfgr        AS manufacturer,
    p_brand       AS brand,
    p_type        AS part_type,
    p_size        AS part_size,
    p_container   AS container,
    p_retailprice AS retail_price
FROM {SILVER}.part
""")

print(f"Created {GOLD}.dim_part")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.dim_date
COMMENT 'Order-date dimension used by Finance analytics'
AS
SELECT DISTINCT
    CAST(date_format(o_orderdate, 'yyyyMMdd') AS INT) AS date_key,
    o_orderdate AS full_date,
    YEAR(o_orderdate) AS year,
    QUARTER(o_orderdate) AS quarter,
    MONTH(o_orderdate) AS month,
    date_format(o_orderdate, 'MMMM') AS month_name,
    DAYOFMONTH(o_orderdate) AS day_of_month,
    dayofweek(o_orderdate) AS day_of_week,
    CONCAT(
        CAST(YEAR(o_orderdate) AS STRING),
        '-Q',
        CAST(QUARTER(o_orderdate) AS STRING)
    ) AS year_quarter
FROM {SILVER}.orders
""")

print(f"Created {GOLD}.dim_date")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_lineitem
COMMENT 'Finance line-item fact table; one row per order line'
AS
SELECT
    l.l_orderkey AS order_key,
    l.l_linenumber AS line_number,

    o.o_custkey AS customer_key,
    l.l_suppkey AS supplier_key,
    l.l_partkey AS part_key,

    CAST(date_format(o.o_orderdate, 'yyyyMMdd') AS INT) AS date_key,
    o.o_orderdate AS order_date,

    l.l_quantity AS quantity,
    l.l_extendedprice AS extended_price,
    l.l_discount AS discount,
    l.l_tax AS tax,

    CAST(l.l_extendedprice AS DECIMAL(38,4)) AS gross_revenue,

    CAST(
        l.l_extendedprice * l.l_discount
        AS DECIMAL(38,4)
    ) AS discount_amount,

    CAST(
        l.l_extendedprice * (1 - l.l_discount)
        AS DECIMAL(38,4)
    ) AS net_revenue,

    CAST(
        l.l_extendedprice * (1 - l.l_discount) * l.l_tax
        AS DECIMAL(38,4)
    ) AS tax_amount,

    CAST(
        l.l_extendedprice * (1 - l.l_discount) * (1 + l.l_tax)
        AS DECIMAL(38,4)
    ) AS net_revenue_with_tax

FROM {SILVER}.lineitem l
INNER JOIN {SILVER}.orders o
    ON l.l_orderkey = o.o_orderkey
""")

print(f"Created {GOLD}.fact_lineitem")


In [ ]:
# Show the finished Gold objects.
display(
    spark.sql(f"SHOW TABLES IN {GOLD}")
    .select("tableName", "isTemporary")
    .orderBy("tableName")
)


## 2. Sanity checks before answering business questions

These are lightweight checks for this notebook. Formal cross-layer validation/monitoring can still be owned by the validation role.


In [ ]:
# 2.1 Fact row count must equal clean Silver lineitem row count.
silver_lineitems = spark.table(f"{SILVER}.lineitem").count()
gold_lineitems = spark.table(f"{GOLD}.fact_lineitem").count()

print(f"Silver lineitem rows: {silver_lineitems:,}")
print(f"Gold fact rows      : {gold_lineitems:,}")

assert gold_lineitems == silver_lineitems, (
    "fact_lineitem row count differs from Silver lineitem; "
    "the orders join may have dropped or duplicated rows."
)


In [ ]:
# 2.2 (order_key, line_number) must remain unique.
duplicate_fact_keys = spark.sql(f"""
SELECT COUNT(*) AS duplicate_groups
FROM (
    SELECT order_key, line_number
    FROM {GOLD}.fact_lineitem
    GROUP BY order_key, line_number
    HAVING COUNT(*) > 1
)
""").first()["duplicate_groups"]

print(f"Duplicate fact keys: {duplicate_fact_keys}")
assert duplicate_fact_keys == 0, "Duplicate fact keys found."


In [ ]:
# 2.3 Dimension keys must remain unique.
dimension_checks = {
    "dim_customer": "customer_key",
    "dim_supplier": "supplier_key",
    "dim_part": "part_key",
    "dim_date": "date_key",
}

for table_name, key_col in dimension_checks.items():
    duplicate_groups = spark.sql(f"""
        SELECT COUNT(*) AS duplicate_groups
        FROM (
            SELECT {key_col}
            FROM {GOLD}.{table_name}
            GROUP BY {key_col}
            HAVING COUNT(*) > 1
        )
    """).first()["duplicate_groups"]

    print(f"{table_name:<15} duplicate key groups = {duplicate_groups}")
    assert duplicate_groups == 0, f"Duplicate keys found in {table_name}."


In [ ]:
# 2.4 Derived revenue in Gold must equal the same formula calculated directly from Silver.
revenue_check = spark.sql(f"""
SELECT
    (
        SELECT SUM(CAST(l_extendedprice * (1 - l_discount) AS DECIMAL(38,4)))
        FROM {SILVER}.lineitem
    ) AS silver_net_revenue,

    (
        SELECT SUM(net_revenue)
        FROM {GOLD}.fact_lineitem
    ) AS gold_net_revenue
""").first()

silver_net = revenue_check["silver_net_revenue"]
gold_net = revenue_check["gold_net_revenue"]
difference = abs(silver_net - gold_net)

print(f"Silver net revenue: {silver_net}")
print(f"Gold net revenue  : {gold_net}")
print(f"Difference        : {difference}")

assert difference <= Decimal("0.01"), "Silver and Gold net revenue do not reconcile."


# Finance business questions

The reusable views below make the answers easy to query from the notebook, a Databricks dashboard, or the final demo.


## Q1 — Top 10 customers by net revenue

**Question:** Which 10 customers generate the most net revenue, and what share of total revenue do they account for?

Revenue definition: `extended_price × (1 − discount)`.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_q1_top_customers AS

WITH customer_revenue AS (
    SELECT
        customer_key,
        SUM(net_revenue) AS customer_net_revenue
    FROM {GOLD}.fact_lineitem
    GROUP BY customer_key
),

ranked AS (
    SELECT
        customer_key,
        customer_net_revenue,
        ROW_NUMBER() OVER (
            ORDER BY customer_net_revenue DESC, customer_key
        ) AS revenue_rank
    FROM customer_revenue
),

totals AS (
    SELECT
        SUM(customer_net_revenue) AS total_net_revenue,
        SUM(
            CASE WHEN revenue_rank <= 10
                 THEN customer_net_revenue
                 ELSE 0
            END
        ) AS top_10_net_revenue
    FROM ranked
)

SELECT
    r.revenue_rank,
    r.customer_key,
    c.customer_name,
    c.market_segment,
    r.customer_net_revenue,

    CAST(
        100.0 * r.customer_net_revenue / t.total_net_revenue
        AS DECIMAL(10,4)
    ) AS customer_share_pct,

    CAST(
        100.0 * t.top_10_net_revenue / t.total_net_revenue
        AS DECIMAL(10,4)
    ) AS top_10_combined_share_pct

FROM ranked r
JOIN {GOLD}.dim_customer c
    ON r.customer_key = c.customer_key
CROSS JOIN totals t
WHERE r.revenue_rank <= 10
ORDER BY r.revenue_rank
""")

q1 = spark.table(f"{GOLD}.vw_q1_top_customers")
display(q1)

top10_share = q1.select("top_10_combined_share_pct").first()[0]
print(f"Top 10 customers account for {top10_share}% of total net revenue.")


In [ ]:
# Q1 visualisation: top 10 customers by net revenue.
import matplotlib.pyplot as plt

q1_pdf = (
    spark.table(f"{GOLD}.vw_q1_top_customers")
    .orderBy(F.desc("customer_net_revenue"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(q1_pdf["customer_name"], q1_pdf["customer_net_revenue"])
ax.invert_yaxis()
ax.set_title("Top 10 Customers by Net Revenue")
ax.set_xlabel("Net revenue")
ax.set_ylabel("Customer")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## Q2 — 1997 revenue three ways

**Question:** Report 1997 revenue as:
1. gross extended price;
2. net of discount;
3. net of discount and tax.

The project-wide reporting metric used elsewhere in this notebook is **net revenue excluding tax**. It reflects the sales amount after discounts while keeping tax separate. Gross extended price ignores discounts, while adding tax mixes the company's sales with tax collected.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_q2_1997_revenue AS

WITH revenue AS (
    SELECT
        SUM(gross_revenue) AS gross_extended_price,
        SUM(net_revenue) AS net_of_discount,
        SUM(net_revenue_with_tax) AS net_of_discount_and_tax
    FROM {GOLD}.fact_lineitem
    WHERE YEAR(order_date) = 1997
)

SELECT
    gross_extended_price,
    net_of_discount,
    net_of_discount_and_tax,

    gross_extended_price - net_of_discount
        AS discount_effect_amount,

    CAST(
        100.0 * (gross_extended_price - net_of_discount)
        / gross_extended_price
        AS DECIMAL(10,4)
    ) AS discount_effect_pct,

    net_of_discount_and_tax - net_of_discount
        AS tax_effect_amount,

    CAST(
        100.0 * (net_of_discount_and_tax - net_of_discount)
        / net_of_discount
        AS DECIMAL(10,4)
    ) AS tax_effect_pct,

    net_of_discount_and_tax - gross_extended_price
        AS final_vs_gross_difference

FROM revenue
""")

q2 = spark.table(f"{GOLD}.vw_q2_1997_revenue")
display(q2)


In [ ]:
import matplotlib.pyplot as plt

# Q2 visualisation: compare the three 1997 revenue definitions.
q2_row = spark.table(f"{GOLD}.vw_q2_1997_revenue").first()

labels = [
    "Gross extended price",
    "Net of discount",
    "Net of discount + tax",
]
values = [
    float(q2_row["gross_extended_price"]),
    float(q2_row["net_of_discount"]),
    float(q2_row["net_of_discount_and_tax"]),
]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values)
ax.set_title("1997 Revenue Under Three Definitions")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.25)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,.0f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()


## Q3 — Average Order Value by market segment and year

**Question:** Which market segment has the highest average order value, and does that change year over year?

Chosen definition:

`AOV = SUM(net_revenue) / COUNT(DISTINCT order_key)`

Using `AVG(net_revenue)` would be incorrect because that would measure an average **line-item** value, not an average **order** value.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_q3_aov_by_segment_year AS

SELECT
    YEAR(f.order_date) AS year,
    c.market_segment,
    COUNT(DISTINCT f.order_key) AS order_count,
    SUM(f.net_revenue) AS total_net_revenue,

    CAST(
        SUM(f.net_revenue) / COUNT(DISTINCT f.order_key)
        AS DECIMAL(38,4)
    ) AS average_order_value

FROM {GOLD}.fact_lineitem f
JOIN {GOLD}.dim_customer c
    ON f.customer_key = c.customer_key

GROUP BY
    YEAR(f.order_date),
    c.market_segment
""")

display(
    spark.table(f"{GOLD}.vw_q3_aov_by_segment_year")
    .orderBy("year", F.desc("average_order_value"))
)


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_q3_yearly_winner AS

WITH ranked AS (
    SELECT
        year,
        market_segment,
        order_count,
        total_net_revenue,
        average_order_value,
        DENSE_RANK() OVER (
            PARTITION BY year
            ORDER BY average_order_value DESC
        ) AS aov_rank
    FROM {GOLD}.vw_q3_aov_by_segment_year
)

SELECT
    year,
    market_segment,
    order_count,
    total_net_revenue,
    average_order_value
FROM ranked
WHERE aov_rank = 1
ORDER BY year, market_segment
""")

q3_winners = spark.table(f"{GOLD}.vw_q3_yearly_winner")
display(q3_winners)

print("Highest-AOV segment(s) by year:")
for row in q3_winners.collect():
    print(
        f"{row['year']}: {row['market_segment']} "
        f"(AOV={row['average_order_value']})"
    )


In [ ]:
import matplotlib.pyplot as plt

# Q3 visualisation: AOV trend for every market segment.
q3_pdf = (
    spark.table(f"{GOLD}.vw_q3_aov_by_segment_year")
    .orderBy("year", "market_segment")
    .toPandas()
)

pivot = q3_pdf.pivot(
    index="year",
    columns="market_segment",
    values="average_order_value"
).astype(float)

ax = pivot.plot(marker="o", figsize=(10, 6))
ax.set_title("Average Order Value by Market Segment and Year")
ax.set_xlabel("Year")
ax.set_ylabel("Average order value")
ax.grid(alpha=0.25)
ax.legend(title="Market segment")
plt.tight_layout()
plt.show()


## Q4 — Top 5 suppliers

**Question:** Which 5 suppliers generate the most net revenue, and what is their average discount?

`average_discount_pct` is the arithmetic mean of the line-item discount rates for each supplier.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_q4_top_suppliers AS

WITH supplier_metrics AS (
    SELECT
        supplier_key,
        SUM(net_revenue) AS supplier_net_revenue,
        AVG(discount) AS average_discount
    FROM {GOLD}.fact_lineitem
    GROUP BY supplier_key
),

ranked AS (
    SELECT
        supplier_key,
        supplier_net_revenue,
        average_discount,
        ROW_NUMBER() OVER (
            ORDER BY supplier_net_revenue DESC, supplier_key
        ) AS revenue_rank
    FROM supplier_metrics
)

SELECT
    r.revenue_rank,
    r.supplier_key,
    s.supplier_name,
    r.supplier_net_revenue,
    CAST(100.0 * r.average_discount AS DECIMAL(10,4))
        AS average_discount_pct

FROM ranked r
JOIN {GOLD}.dim_supplier s
    ON r.supplier_key = s.supplier_key

WHERE r.revenue_rank <= 5
ORDER BY r.revenue_rank
""")

q4 = spark.table(f"{GOLD}.vw_q4_top_suppliers")
display(q4)


In [ ]:
import matplotlib.pyplot as plt

# Q4 visualisation: revenue bars with average discount shown above each supplier.
q4_pdf = (
    spark.table(f"{GOLD}.vw_q4_top_suppliers")
    .orderBy("revenue_rank")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(q4_pdf["supplier_name"], q4_pdf["supplier_net_revenue"])

ax.set_title("Top 5 Suppliers by Net Revenue")
ax.set_xlabel("Supplier")
ax.set_ylabel("Net revenue")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.25)

for bar, discount_pct in zip(bars, q4_pdf["average_discount_pct"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"avg discount {float(discount_pct):.2f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()


## 3. Final summary for the demo

Run this cell near the end of the presentation to show the key answers in compact form.


In [ ]:
print("=" * 70)
print("Q1 — TOP 10 CUSTOMERS")
display(spark.table(f"{GOLD}.vw_q1_top_customers"))

print("=" * 70)
print("Q2 — 1997 REVENUE THREE WAYS")
display(spark.table(f"{GOLD}.vw_q2_1997_revenue"))

print("=" * 70)
print("Q3 — HIGHEST-AOV SEGMENT BY YEAR")
display(spark.table(f"{GOLD}.vw_q3_yearly_winner"))

print("=" * 70)
print("Q4 — TOP 5 SUPPLIERS")
display(spark.table(f"{GOLD}.vw_q4_top_suppliers"))


## Gold objects created

**Tables**
- `dim_customer`
- `dim_supplier`
- `dim_part`
- `dim_date`
- `fact_lineitem`

**Views**
- `vw_q1_top_customers`
- `vw_q2_1997_revenue`
- `vw_q3_aov_by_segment_year`
- `vw_q3_yearly_winner`
- `vw_q4_top_suppliers`

All financial revenue questions use the same core definition:

`net_revenue = extended_price × (1 − discount)`
